In [ ]:
# NOTEBOOK 01 — BUILD DATASET
# descarga, conversión y construcción del dataset de contaminantes

'''
este notebook ejecuta el proceso completo de construcción del dataset de contaminantes a partir de los
archivos Parquet oficiales de la European Environment Agency (EEA)

los CSV iniciales contienen únicamente las URLs de los Parquet originales
aquí realizo:
    carga de los CSV con las URLs (RAW)
    descarga de todos los archivos Parquet desde las URLs oficiales
    conversión de cada Parquet a CSV para evitar problemas de compatibilidad y memoria
    carga y concatenación de todos los CSV de cada contaminante
    generación de un único dataframe por contaminante
    conversión final a Parquet limpio, plano y homogéneo (uno por contaminante)

IMPORTANTE:
este notebook solo se ejecuta una vez, ya que produce los archivos base que se usarán
en los siguientes notebooks del proyecto
'''

In [2]:
# importaciones de librerías

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import requests
from io import BytesIO
from tqdm import tqdm  # con esto añado una barra de progreso que me parece cuqui
import glob
import gc
import pyarrow as pa
import pyarrow.parquet as pq



In [2]:
'''esto lo meto porque es una configuración visual de pandas para que no me oculte columnas
cuando haga df.head() o df.info():'''
pd.set_option('display.max_columns', None)


In [ ]:
# carga de datasets

'''
DATA DE CONTAMINANTES:
-data_pm25 → datos de concentración de PM2.5 (partículas finas) por estación y fecha
-data_pm10 → Datos de concentración de PM10 (partículas gruesas)
-data_no2 → Datos de dióxido de nitrógeno (NO2), asociado al tráfico urbano
-data_o3 → Datos de ozono troposférico (O3), contaminante secundario con fuerte estacionalidad
-data_so2 → Datos de dióxido de azufre (SO2), relacionado con industria y combustibles fósiles
-data_co → Datos de monóxido de carbono (CO), indicador de combustión incompleta
-data_c6h6 → Datos de benceno (C6H6), compuesto orgánico volátil tóxico
-data_bc → Datos de carbono negro (Black Carbon), asociado al tráfico y biomasa

OTROS DATAS:
-df_hospi → Hospitalizaciones por causas respiratorias en España
-df_mortality → Mortalidad por enfermedades respiratorias
-df_population → Población por provincia/comunidad (para tasas por 100k habitantes)
-df_stations → Metadata de estaciones de calidad del aire (ubicación, tipo, zona, etc.)
'''

data_pm25 = pd.read_csv("../data/ParquetFilesUrls_PM25.csv")
data_pm10 = pd.read_csv("../data/ParquetFilesUrls_PM10.csv")
data_no2  = pd.read_csv("../data/ParquetFilesUrls_NO2.csv")
data_o3   = pd.read_csv("../data/ParquetFilesUrls_O3.csv")
data_so2  = pd.read_csv("../data/ParquetFilesUrls_SO2.csv")
data_co   = pd.read_csv("../data/ParquetFilesUrls_CO.csv")
data_c6h6 = pd.read_csv("../data/ParquetFilesUrls_C6H6.csv")
data_bc   = pd.read_csv("../data/ParquetFilesUrls_BC.csv")

df_hospi = pd.read_csv("../data/hospitalizations_spain.csv")
df_mortality = pd.read_csv("../data/mortality_respiratory_spain.csv")
df_population = pd.read_csv("../data/population_spain.csv")
df_stations = pd.read_csv("../data/stations_metadata_spain.csv")

'\nNOTA IMPORTANTE SOBRE LOS DATOS DE CONTAMINACIÓN:\nEste notebook procesa los datos oficiales de calidad del aire en España\nproporcionados por la European Environment Agency (EEA).\n\nLos CSV iniciales NO contienen datos de contaminación, sino únicamente\nlas URLs de los archivos Parquet donde están los datos reales.\n\nFlujo de trabajo:\n1) Cargar CSV con URLs (RAW)\n2) Extraer URLs\n3) Descargar Parquet desde Python\n4) Convertir Parquet → CSV\n5) Cargar CSV convertidos y generar df_pm25, df_pm10, etc.\n'

In [ ]:
# compruebo a ver qué sale

for name, df in {
    "PM25": data_pm25,
    "PM10": data_pm10,
    "NO2": data_no2,
    "O3": data_o3,
    "SO2": data_so2,
    "CO": data_co,
    "C6H6": data_c6h6,
    "BC": data_bc,
    "Hospitalizaciones": df_hospi,
    "Mortalidad": df_mortality,
    "Población": df_population,
    "Estaciones": df_stations
}.items():
    print(name, df.shape)


In [ ]:
# efectivamente, sale mal

In [ ]:
# para lo siguiente que voy a hacer necesito instalar pyarrow y fastparquet, que son imprescindibles para leer archivos Parquet
#!pip install pyarrow

In [ ]:
#!pip install pyarrow fastparquet

In [ ]:
# tengo que crear una carpeta para guardar los CSV convertidos
os.makedirs("../data/pollutants/raw", exist_ok=True)

# llamo a la carpeta dentro de data 'raw' porque es porque es el nombre estándar
# en proyectos de ciencia de datos para indicar datos crudos, sin procesar

In [ ]:
# para poder hacer la barra de progreso en la siguiente parte
#!pip install tqdm

In [ ]:
# ahora creo una función para descargar los parquet y guardarlos en CSV

'''
ya están cargadas:
import requests
from io import BytesIO
from tqdm import tqdm
'''

def parquet_to_csv(url_list, output_prefix):
    print(f"\nIniciando descarga y conversión de {output_prefix.upper()}...\n")
    for i, url in enumerate(tqdm(url_list, desc=f"Descargando {output_prefix}")):
        try:
            response = requests.get(url)
            response.raise_for_status()
            content = BytesIO(response.content)
            # intento 1 con pyarrow
            try:
                df = pd.read_parquet(content, engine="pyarrow")
            except:
                # intento 2 con fastparquet
                content.seek(0)
                df = pd.read_parquet(content, engine="fastparquet")

            output_path = f"../data/pollutants/raw/{output_prefix}_{i+1}.csv"
            df.to_csv(output_path, index=False)

        except Exception as e:
            print(f"Error con {url}: {e}")

    print(f"\nConversión de {output_prefix.upper()} completada ({len(url_list)} archivos procesados).\n")


In [ ]:
# extraigo las URLs de cada CSV

urls_pm25 = data_pm25.iloc[:, 0].dropna().tolist()
urls_pm10 = data_pm10.iloc[:, 0].dropna().tolist()
urls_no2  = data_no2.iloc[:, 0].dropna().tolist()
urls_o3   = data_o3.iloc[:, 0].dropna().tolist()
urls_so2  = data_so2.iloc[:, 0].dropna().tolist()
urls_co   = data_co.iloc[:, 0].dropna().tolist()
urls_c6h6 = data_c6h6.iloc[:, 0].dropna().tolist()
urls_bc   = data_bc.iloc[:, 0].dropna().tolist()


In [ ]:
# descarga y conversión de parquet a csv

parquet_to_csv(urls_pm25, "pm25")
parquet_to_csv(urls_pm10, "pm10")
parquet_to_csv(urls_no2, "no2")
parquet_to_csv(urls_o3, "o3")
parquet_to_csv(urls_so2, "so2")
parquet_to_csv(urls_co, "co")
parquet_to_csv(urls_c6h6, "c6h6")
parquet_to_csv(urls_bc, "bc")


In [ ]:
# estado del proceso, cuántos CSV se han generado

def contar_csv(prefix):
    '''
    cuenta cuántos archivos CSV existen en la carpeta data/raw cuyo nombre empieza por el prefijo
    indicado (por ejemplo: 'pm25', 'no2', etc.)
    sirve para verificar cuántos archivos se han descargado y convertido para cada contaminante
    '''
    archivos = [f for f in os.listdir("data/raw") if f.startswith(prefix)]
    print(f"{prefix.upper()}: {len(archivos)} CSV generados")

contar_csv("pm25")
contar_csv("pm10")
contar_csv("no2")
contar_csv("o3")
contar_csv("so2")
contar_csv("co")
contar_csv("c6h6")
contar_csv("bc")


PM25: 431 CSV generados
PM10: 705 CSV generados
NO2: 613 CSV generados
O3: 510 CSV generados
SO2: 525 CSV generados
CO: 311 CSV generados
C6H6: 194 CSV generados
BC: 0 CSV generados


In [ ]:
'''
el contaminante BC (Black Carbon) no se incluye en el análisis porque la EEA no proporciona datos
disponibles para el periodo 2011-2019
su ausencia no afecta al proyecto: no forma parte de las hipótesis principales y los contaminantes más
relevantes para salud respiratoria (PM2.5, PM10, NO₂ y O₃) sí están completos
BC es interesante, pero no es imprescindible para evaluar impactos en salud pública ni altera la calidad del análisis
'''

In [ ]:
# creo una función para cargar todos los csv convertidos

'''
ya está cargada:
import glob
'''

def load_all_csv(prefix):
    '''
    busca todos los CSV de un contaminante, los carga uno por uno, los concatena en un único
    dataframe gigante y te devuelve ese dataframe final
    '''
    files = glob.glob(f"../data/pollutants/raw/{prefix}_*.csv")
    dfs = [pd.read_csv(f, engine="python", dtype=str) for f in files]
    return pd.concat(dfs, ignore_index=True)


In [ ]:
# probemos a ver qué sale:

df_pm25 = load_all_csv("pm25")
df_pm25.shape


In [ ]:
'''
debido al volumen extremo de datos, cada contaminante debe procesarse por separado, cargar todos a la vez
saturaba la RAM, así que el flujo correcto es:
    cargar un contaminante
    convertir sus CSV a un único dataframe
    guardarlo como Parquet limpio y comprimido
    liberar memoria y pasar al siguiente
    
los problemas iniciales con Parquet se debían a los archivos originales de la EEA: múltiples Parquet por
contaminante, esquemas complejos y compresión pesada
los Parquet que genero aquí son planos, homogéneos y 100% compatibles con pandas, por lo que no presentan esos problemas
'''

In [ ]:
# df_pm25.dtypes

In [ ]:
# Creo la carpeta processed
os.makedirs("../data/pollutants/processed", exist_ok=True)


In [ ]:
# compruebo que se creó la carpeta
os.listdir("data")


In [ ]:
'''
ya están cargadas:
import pyarrow as pa
import pyarrow.parquet as pq
'''

# Convertir el DataFrame pandas a una Arrow Table
table = pa.Table.from_pandas(df_pm25, preserve_index=False)

# Guardar con pyarrow (sin errores)
pq.write_table(table, "../data/pollutants/processed/pm25.parquet")

# Liberar memoria
del df_pm25, table
gc.collect()


In [3]:
# veo si se guardó:
os.listdir("../data/pollutants/processed")


['c6h6.parquet',
 'co.parquet',
 'no2.parquet',
 'o3.parquet',
 'pm10.parquet',
 'pm25.parquet',
 'so2.parquet']

In [ ]:
# pruebo que se puede cargar
df_pm25 = pd.read_parquet("../data/pollutants/processed/pm25.parquet")
df_pm25.head()


In [ ]:
os.listdir("data")


In [ ]:
# AHORA VAMOS CON EL SIGUIENTE CONTAMINANTE: PM10

In [ ]:
df_pm10 = load_all_csv("pm10")
df_pm10.dtypes


In [ ]:
table = pa.Table.from_pandas(df_pm10, preserve_index=False)
pq.write_table(table, "../data/pollutants/processed/pm10.parquet")
del df_pm10, table
gc.collect()


In [ ]:
# comprobamos si está:
os.listdir("../data/pollutants/processed")


In [ ]:
# siguiente contaminante: NO2

df_no2 = load_all_csv("no2")
table = pa.Table.from_pandas(df_no2, preserve_index=False)
pq.write_table(table, "../data/pollutants/processed/no2.parquet")
del df_no2, table
gc.collect()


In [ ]:
# O3

df_o3 = load_all_csv("o3")
table = pa.Table.from_pandas(df_o3, preserve_index=False)
pq.write_table(table, "../data/pollutants/processed/o3.parquet")
del df_o3, table
gc.collect()


In [ ]:
# SO2

df_so2 = load_all_csv("so2")
table = pa.Table.from_pandas(df_so2, preserve_index=False)
pq.write_table(table, "../data/pollutants/processed/so2.parquet")
del df_so2, table
gc.collect()


In [ ]:
# CO

df_co = load_all_csv("co")
table = pa.Table.from_pandas(df_co, preserve_index=False)
pq.write_table(table, "../data/pollutants/processed/co.parquet")
del df_co, table
gc.collect()


19

In [ ]:
# C6H6

df_c6h6 = load_all_csv("c6h6")
table = pa.Table.from_pandas(df_c6h6, preserve_index=False)
pq.write_table(table, "../data/pollutants/processed/c6h6.parquet")
del df_c6h6, table
gc.collect()

In [ ]:
# ignoro el df de Black Carbon porque hemos visto que no había información

In [5]:
# comprobamos si están todos:
os.listdir("../data/pollutants/processed")

['c6h6.parquet',
 'co.parquet',
 'no2.parquet',
 'o3.parquet',
 'pm10.parquet',
 'pm25.parquet',
 'so2.parquet']

In [6]:
os.path.getsize("../data/pollutants/processed/o3.parquet") / (1024**2)

534.5793514251709

In [ ]:
# CIERRE DEL NOTEBOOK 01

'''
en este notebook he ejecutado todo el proceso de construcción del dataset de contaminantes a partir de
los archivos Parquet oficiales de la EEA

el proceso completo ha sido:
    cargar los CSV iniciales que contienen únicamente las urls de los Parquet
    descargar todos los Parquet originales desde las urls oficiales
    convertir cada Parquet a CSV para evitar problemas de compatibilidad y memoria
    cargar y concatenar cientos de CSV por contaminante
    generar un único dataframe por contaminante
    guardar cada contaminante en un Parquet limpio, plano y homogéneo

debido al volumen extremo de datos, cada contaminante se ha procesado por separado para evitar saturar la RAM
los Parquet generados aquí no tienen nada que ver con los Parquet originales de la EEA: son simples y
totalmente compatibles con pandas, por lo que servirán como base estable para el resto del proyecto

este notebook solo debe ejecutarse una vez
a partir de aquí, los notebooks 02, 03 y 04 trabajarán directamente con los Parquet limpios generados en este paso

'''